# SHOT: YOLOv8n Tennis Ball Detector

**데이터**: TrackNet 19,835장 + Roboflow 9,737장 = ~30,000장
**모델**: YOLOv8n (640x640 입력)
**출력**: ball_detector_yolo.onnx

**Setup:**
1. Add Data: `datasets/faultfoot` (TrackNet)
2. Settings → Accelerator → GPU T4 x2
3. Run All

In [ ]:
# 셀 0: 설치
!pip install ultralytics roboflow -q

In [ ]:
# 셀 1: Roboflow 데이터셋 다운로드
from roboflow import Roboflow

rf = Roboflow(api_key="TdYIJQTWeBtbsT2vq0Sm")
project = rf.workspace("tennisball-3eqxr").project("tennis-ball-detection-qaxae")
rf_dataset = project.version(1).download("yolov8", location="/kaggle/working/roboflow_ball")

import os
for split in ['train', 'valid', 'test']:
    p = f"/kaggle/working/roboflow_ball/{split}/images"
    if os.path.exists(p):
        print(f"{split}: {len(os.listdir(p))} images")

In [ ]:
# 셀 2: TrackNet → YOLO 변환
import csv, json, os, re
from pathlib import Path

TRACKNET_JSON = "/kaggle/input/datasets/faultfoot/balldata/ball_combined.json"
TRACKNET_FRAMES = "/kaggle/input/datasets/faultfoot/balldata/frames"
TRACKNET_YOLO = "/kaggle/working/tracknet_yolo"

# Check
if not os.path.exists(TRACKNET_JSON):
    print(f"[MISSING] {TRACKNET_JSON}")
    print("Available:")
    for item in os.listdir("/kaggle/input"):
        full = os.path.join("/kaggle/input", item)
        print(f"  {item}/")
        if os.path.isdir(full):
            for sub in os.listdir(full)[:5]:
                print(f"    {sub}")
else:
    print(f"[OK] TrackNet data found")

with open(TRACKNET_JSON) as f:
    annotations = json.load(f)
print(f"Total annotations: {len(annotations)}")

# Group by video for split
from collections import defaultdict
import numpy as np

video_frames = defaultdict(list)
for ann in annotations:
    match = re.match(r'(.+?)_frame_', ann['image'])
    vid_id = match.group(1) if match else 'unknown'
    video_frames[vid_id].append(ann)

video_ids = sorted(video_frames.keys())
np.random.seed(42)
np.random.shuffle(video_ids)
split_idx = int(len(video_ids) * 0.85)

train_vids = set(video_ids[:split_idx])
val_vids = set(video_ids[split_idx:])

# Create YOLO structure
for split in ['train', 'val']:
    os.makedirs(f"{TRACKNET_YOLO}/{split}/images", exist_ok=True)
    os.makedirs(f"{TRACKNET_YOLO}/{split}/labels", exist_ok=True)

BALL_W, BALL_H = 0.015, 0.028  # ~20px in 1280x720
train_count = val_count = 0

for ann in annotations:
    match = re.match(r'(.+?)_frame_', ann['image'])
    vid_id = match.group(1) if match else 'unknown'
    split = 'train' if vid_id in train_vids else 'val'
    
    img_src = os.path.join(TRACKNET_FRAMES, ann['image'])
    if not os.path.exists(img_src):
        continue
    
    img_dst = f"{TRACKNET_YOLO}/{split}/images/tn_{ann['image']}"
    if not os.path.exists(img_dst):
        os.symlink(img_src, img_dst)
    
    stem = Path(ann['image']).stem
    lbl_path = f"{TRACKNET_YOLO}/{split}/labels/tn_{stem}.txt"
    
    if ann['visibility'] > 0 and ann['x'] >= 0:
        with open(lbl_path, 'w') as f:
            f.write(f"0 {ann['x']:.6f} {ann['y']:.6f} {BALL_W} {BALL_H}\n")
    else:
        with open(lbl_path, 'w') as f:
            pass  # empty = negative sample
    
    if split == 'train': train_count += 1
    else: val_count += 1

print(f"TrackNet YOLO: train={train_count}, val={val_count}")

In [ ]:
# 셀 3: 데이터셋 병합
import shutil

MERGED = "/kaggle/working/merged_ball"
for split in ['train', 'val']:
    os.makedirs(f"{MERGED}/{split}/images", exist_ok=True)
    os.makedirs(f"{MERGED}/{split}/labels", exist_ok=True)

def safe_link(src, dst):
    if os.path.exists(dst): return
    try:
        os.symlink(src, dst)
    except:
        shutil.copy2(src, dst)

counts = {}

# TrackNet
for split in ['train', 'val']:
    img_dir = f"{TRACKNET_YOLO}/{split}/images"
    lbl_dir = f"{TRACKNET_YOLO}/{split}/labels"
    c = 0
    for f in os.listdir(img_dir):
        safe_link(os.path.join(img_dir, f), f"{MERGED}/{split}/images/{f}")
        lbl = os.path.splitext(f)[0] + '.txt'
        lbl_src = os.path.join(lbl_dir, lbl)
        if os.path.exists(lbl_src):
            safe_link(lbl_src, f"{MERGED}/{split}/labels/{lbl}")
        c += 1
    counts[f'tracknet_{split}'] = c

# Roboflow (train→train, valid→val, test→train)
RF = "/kaggle/working/roboflow_ball"
rf_map = {'train': 'train', 'valid': 'val', 'test': 'train'}
for rf_split, my_split in rf_map.items():
    img_dir = f"{RF}/{rf_split}/images"
    lbl_dir = f"{RF}/{rf_split}/labels"
    if not os.path.exists(img_dir): continue
    c = 0
    for f in os.listdir(img_dir):
        safe_link(os.path.join(img_dir, f), f"{MERGED}/{my_split}/images/rf_{f}")
        lbl = os.path.splitext(f)[0] + '.txt'
        lbl_src = os.path.join(lbl_dir, lbl)
        if os.path.exists(lbl_src):
            safe_link(lbl_src, f"{MERGED}/{my_split}/labels/rf_{lbl}")
        c += 1
    counts[f'roboflow_{rf_split}->{my_split}'] = c

# Count
train_total = len(os.listdir(f"{MERGED}/train/images"))
val_total = len(os.listdir(f"{MERGED}/val/images"))

print("=== Merged Dataset ===")
for k, v in sorted(counts.items()):
    print(f"  {k}: {v}")
print(f"\nTotal train: {train_total}")
print(f"Total val: {val_total}")
print(f"Grand total: {train_total + val_total}")

# Write data.yaml
yaml_text = f"""path: {MERGED}
train: train/images
val: val/images

nc: 1
names: ['ball']
"""
with open(f"{MERGED}/data.yaml", 'w') as f:
    f.write(yaml_text)
print("data.yaml written")

In [ ]:
# 셀 4: YOLOv8n 학습
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # pretrained nano

results = model.train(
    data=f"{MERGED}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,
    patience=15,
    workers=4,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    project='/kaggle/working/models',
    name='ball_yolov8n',
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
)

print("\n=== Training Complete ===")
print(f"Best model: /kaggle/working/models/ball_yolov8n/weights/best.pt")

In [ ]:
# 셀 5: 검증
model = YOLO('/kaggle/working/models/ball_yolov8n/weights/best.pt')
metrics = model.val(data=f"{MERGED}/data.yaml", imgsz=640, device=0)

print(f"\n=== Validation Results ===")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

In [ ]:
# 셀 6: ONNX 변환
model = YOLO('/kaggle/working/models/ball_yolov8n/weights/best.pt')

model.export(
    format='onnx',
    imgsz=640,
    opset=18,
    simplify=True,
    dynamic=False,
)

import shutil
onnx_src = '/kaggle/working/models/ball_yolov8n/weights/best.onnx'
onnx_dst = '/kaggle/working/models/ball_detector_yolo.onnx'
shutil.copy2(onnx_src, onnx_dst)

size_mb = os.path.getsize(onnx_dst) / 1024 / 1024
print(f"\nExported: {onnx_dst} ({size_mb:.2f} MB)")

# Validate ONNX
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession(onnx_dst)
inp = session.get_inputs()[0]
out = session.get_outputs()[0]
print(f"Input:  {inp.name} {inp.shape}")
print(f"Output: {out.name} {out.shape}")

dummy = np.random.randn(1, 3, 640, 640).astype(np.float32)
result = session.run(None, {inp.name: dummy})[0]
print(f"Test output shape: {result.shape}")
print("[OK] ONNX validation passed")

In [ ]:
# 셀 7: Output 파일 확인
import os

print("=== Output Files ===")
for root, dirs, files in os.walk('/kaggle/working/models'):
    for f in files:
        full = os.path.join(root, f)
        size = os.path.getsize(full) / 1024 / 1024
        rel = os.path.relpath(full, '/kaggle/working/models')
        if size > 0.1:
            print(f"  {rel:50s} {size:.2f} MB")